<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-05-deterministic-mini-agent/notebook.ipynb)


# Session 5 — A deterministic mini-agent

**Goal:** complete a tool-calling loop with a trace receipt: a loop budget, a repeated call caught, and a safe termination. *Thread: loop engineering.*

Every cell runs offline on `FakeLLM`. Nothing here calls a provider or the network.


In [ ]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

from bootcamp_agent.checks import check, review


## 1. The loop you already have

`answer_question` is a loop with three exits already designed: refuse when retrieval is empty, retry once on a broken contract, refuse again if the retry fails. The trace is what it did, in order.

In [ ]:
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

from bootcamp_agent.agent import answer_question
from bootcamp_agent.documents import load_corpus
from bootcamp_agent.llm import FakeLLM

documents = load_corpus(CORPUS_DIR)
question = "What defenses help against prompt injection?"
result = answer_question(question, documents, FakeLLM())
for event in result.trace:
    print(f"[{event.kind}] {event.detail}")


## 2. Exercise: the budget, visible in the trace

**Context.** `answer_question` takes `max_tool_calls`. With this corpus the direct path rarely needs a tool; the point is that the bound exists and the trace shows it.

**Instructions.**

1. Budget 3 is done. Add budget 1 with the same question and corpus.
2. Read both lists. Count the `tool_call` events against the budget.
3. Run the check: it confirms neither trace exceeds its own budget, and that both end in a `decision`.

In [ ]:
# ---------------------------------------------------------------------
# THIS RUNS AS SHIPPED, and passes its check. That is the floor.
# To stand above it: stop on a budget you can see in the trace, not one buried in a constant.
# Change it, re-run the check cell below, and keep what you learn.
# ---------------------------------------------------------------------
traces = {
    3: [e.kind for e in answer_question(question, documents, FakeLLM(), max_tool_calls=3).trace],
    1: [e.kind for e in answer_question(question, documents, FakeLLM(), max_tool_calls=1).trace],
}
for budget, kinds in traces.items():
    print(f"budget={budget}: {kinds}")


**Expected output** (yours may differ in wording, not in shape):

```
budget=3: ['retrieve', 'llm_call', 'decision']
budget=1: ['retrieve', 'llm_call', 'decision']
✅ ch05-e1 passed
```

In [ ]:
check("ch05-e1", traces)

## 3. The tools, and a plan over them

Your loop needs something to call. These are session 4's two tools with their contracts intact, plus a **plan**: the sequence of calls a model would have chosen, written down instead. A scripted plan makes every exit reachable on purpose, so the loop is testable without a model in it.

In [ ]:
from bootcamp_agent.tools import ToolError

# Yesterday's two tools, unchanged in contract. The rate table is pinned so this
# notebook never touches the network, and the ids join on one line so a receipt
# prints on one screen.
RATES = {"USD": {"EUR": 0.92, "BRL": 5.40}}


def list_documents(tag: str | None = None) -> str:
    known = {t for doc in documents for t in doc.tags}
    if tag is None:
        return ", ".join(doc.doc_id for doc in documents)
    if not tag.strip():
        raise ToolError("list_documents: 'tag' must be non-empty when given")
    if tag not in known:
        raise ToolError(f"list_documents: unknown tag {tag!r}; valid tags: {sorted(known)}")
    return ", ".join(doc.doc_id for doc in documents if tag in doc.tags)


def convert_currency(amount: float, source: str, target: str) -> str:
    rates = RATES.get(source, {})
    if target not in rates:
        raise ToolError(f"convert_currency: no rate {source}->{target}; known: {sorted(rates)}")
    return f"{amount} {source} = {amount * rates[target]:.2f} {target}"


tools = {"list_documents": list_documents, "convert_currency": convert_currency}
plan = [
    {"tool": "list_documents", "args": {"tag": "retrieval"}},
    {"tool": "convert_currency", "args": {"amount": 100, "source": "USD", "target": "EUR"}},
    {"tool": "answer", "args": {"text": "rag-basics covers retrieval, and 100 USD is 92.00 EUR."}},
]
for step in plan:
    print(f"{step['tool']:18} {step['args']}")


## 4. Exercise: `run_loop`, and its four exits

**Context.** The loop executes one planned call at a time and returns a receipt. Every run ends in exactly one of four designed states — never in a traceback.

| `stopped_because` | When | `answer` | `refusal` |
|---|---|---|---|
| `answered` | the step's tool is `answer` | its `args['text']` | `None` |
| `repeated_call` | this call equals the one before it | `None` | why |
| `budget` | `budget` calls already recorded, or the plan ran out | `None` | why |
| `tool_error` | the tool raised `ToolError` | `None` | why, naming the tool |

`steps` records executed **tool calls** only, one `{"tool", "args", "result"}` dict each. The `answer` step is a decision, not a call, so it is never a step.

**Instructions.**

1. Exit 1 is done. Add exit 2: stop when `(name, args)` equals `previous`.
2. Add exit 3: stop before the call that would exceed `budget`. Exactly `budget` steps get recorded, never one more.
3. Add exit 4: catch `ToolError` around the call. The refusal names the tool, and the loop does not continue to the next planned call.
4. Every stop that is not `answered` writes a sentence into `refusal`. A caller who reads only the receipt has to know why it ended.

In [ ]:
# ---------------------------------------------------------------------
# THE ONE CELL IN THIS SESSION THAT DOES NOT RUN AS SHIPPED.
# The others are written: run them and you have 100 of 200 marks.
# This is the rest. It is the session's point, so it is the one you write.
# ---------------------------------------------------------------------
from collections.abc import Callable


def receipt(steps, stopped_because, answer=None, refusal=None) -> dict:
    """The four keys, on every exit. Given to you; do not change the shape."""
    return {
        "steps": steps,
        "stopped_because": stopped_because,
        "answer": answer,
        "refusal": refusal,
    }


def run_loop(plan: list[dict], tools: dict[str, Callable], budget: int = 5) -> dict:
    steps: list[dict] = []
    previous = None
    for step in plan:
        name, args = step["tool"], step.get("args", {})
        if name == "answer":  # exit 1, done
            return receipt(steps, "answered", answer=args["text"])
        # TODO(you) exit 2: (name, args) equals `previous` -> "repeated_call"
        # TODO(you) exit 3: `budget` calls already recorded -> "budget"
        # TODO(you) exit 4: wrap the call below in try/except ToolError -> "tool_error"
        result = tools[name](**args)
        steps.append({"tool": name, "args": args, "result": result})
        previous = (name, args)
    return receipt(steps, "budget", refusal="stopped: the plan ran out before an answer")


print(run_loop(plan, tools)["stopped_because"])


**Expected output** (yours may differ in wording, not in shape):

```
answered
✅ ch05-e2 passed
```

In [ ]:
check("ch05-e2", run_loop)

## 5. The receipt, read back

This is the artifact: what was called, with what arguments, what came back, and why the run ended. Nobody has to trust a summary of the run when they can read the run.

In [ ]:
lab = run_loop(plan, tools)
for index, step in enumerate(lab["steps"], 1):
    print(f"{index}. {step['tool']}({step['args']}) -> {step['result']}")
print(f"stopped_because={lab['stopped_because']!r}  answer={lab['answer']!r}")


## 6. Failure injection: a tool that starts refusing

A tool that works in the first cell and fails in the fourth is the normal case, not the exotic one: a rate limit, an expired token, an index rebuild. The loop must end with a refusal the caller can read.

Run this before you finish exit 4, and again after. The difference is the lesson.

In [ ]:
calls = {"n": 0}


def flaky_list_documents(tag: str | None = None) -> str:
    """Answers twice, then refuses. A real tool fails mid-run; this one fails on cue."""
    calls["n"] += 1
    if calls["n"] > 2:
        raise ToolError("list_documents: the corpus index went away mid-run")
    return list_documents(tag)


flaky_plan = [
    {"tool": "list_documents", "args": {"tag": "retrieval"}},
    {"tool": "list_documents", "args": {"tag": "security"}},
    {"tool": "list_documents", "args": {"tag": "evaluation"}},
    {"tool": "answer", "args": {"text": "never reached"}},
]
try:
    injected = run_loop(flaky_plan, {"list_documents": flaky_list_documents})
    print(f"steps={len(injected['steps'])}  stopped_because={injected['stopped_because']!r}")
    print(f"refusal: {injected['refusal']}")
except ToolError as error:
    print(f"the error escaped the loop: {error}")
    print("that is the bug — exit 4 in run_loop turns it into a refusal")


## Exit ticket

One thing that works, one thing that is unclear, your next action.

Homework: write the exit table for a loop you have built or used. If a row is empty, that loop is unfinished. Read `docs/guides/loop-engineering.md`.

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [ ]:
review("ch05")